In [ ]:
!pip install shapely
!pip install geojson
!pip install osmnx
!pip install pyproj
!pip install geopandas
!pip install networkx
!pip install matplotlib==3.1.3 # импортирую конкретную версию, тк есть ошибки при построении графиков
!pip install geopandas mapclassify  # mapclassify для визуализации
!pip install folium
!pip install keplergl

In [ ]:
!pip install numpy==1.23.0

In [ ]:
from shapely.geometry import LineString, Polygon, Point
import matplotlib as mpl
import matplotlib.pyplot as plt
from geojson import Feature, FeatureCollection, dump
import pandas as pd
import geopandas as gpd
from scipy import spatial
import json
import networkx as nx
import numpy as np
import statistics
from statistics import mean
import shapely
import mapclassify  # mapclassify для визуализации
import folium
import osmnx as ox
from geopandas.tools import geocode
import keplergl
from keplergl import KeplerGl

In [ ]:
import os

In [ ]:
os.chdir("C:/Users/danil/Desktop/ИТМО/Сбор и обработка/1 часть_Ланцева/Labs/Lab2")

In [ ]:
def dist(p0, p1):   # функция рассчёта в расстояния 
    a, b = Point(p0), Point(p1)
    dist = a.distance(b)
    return dist

In [ ]:
def pedestrian_graph(minX, minY, maxX, maxY):
    bbox = Polygon([[minX,minY], [minX, maxY], [maxX, maxY], [maxX, minY]])
    graph = ox.graph_from_polygon(bbox, network_type = 'walk')
    edges = ox.graph_to_gdfs(graph)
    features = []
    for e in edges[1]['geometry']:
        coord = list(e.coords)
        for i in range(len(coord)-1):
            a,b = Point(coord[i]), Point(coord[i+1])
            dist = a.distance(b)
            features.append(Feature(geometry = LineString([(coord[i][0], coord[i][1]), 
                                                     (coord[i+1][0], coord[i+1][1])]),
                                                      properties = {'distance': dist}))  
    feature_collection = FeatureCollection(features)
    with open('pedestrian_roads.geojson', 'w', encoding = 'utf-8') as f:
        dump(feature_collection, f, ensure_ascii = False, indent = 4)
    return 'success'

In [ ]:
bbox = Polygon([[minX,minY], [minX, maxY], [maxX, maxY], [maxX, minY]])
graph = ox.graph_from_polygon(bbox, network_type = 'walk')
edges = ox.graph_to_gdfs(graph)
features = []
for e in edges[1]['geometry']:
    coord = list(e.coords)
    for i in range(len(coord)-1):
        a,b = Point(coord[i]), Point(coord[i+1])
        dist = a.distance(b)
        features.append(Feature(geometry = LineString([(coord[i][0], coord[i][1]), 
                                                     (coord[i+1][0], coord[i+1][1])]),
                                                      properties = {'distance': dist}))  
feature_collection = FeatureCollection(features)
#     with open('pedestrian_roads.geojson', 'w', encoding = 'utf-8') as f:
#         dump(feature_collection, f, ensure_ascii = False, indent = 4)
#     return 'success'

In [ ]:
minX, minY, maxX, maxY = 30.209141, 59.945586, 30.335134, 59.982592 
# minX, minY, maxX, maxY = 30.335582, 59.812626, 30.440486, 59.920583

In [ ]:
G = pedestrian_graph(minX, minY, maxX, maxY)

In [ ]:
with open('pedestrian_roads.geojson', encoding = 'utf-8', errors='ignore') as json_file:
    data = json.load(json_file)['features']

In [ ]:
G = nx.DiGraph()
for edge in data:
  coord= edge['geometry']['coordinates']
  distance = edge['properties']['distance']
  G.add_edge((coord[0][0], coord[0][1]), (coord[1][0], coord[1][1]), weight = distance)
del edge, coord, distance, data

In [ ]:
def search_nearest_point(Tree_Graph, point):
  sub = Tree_Graph.query([point], 1)
  id_node = sub[1].tolist()[0]
  closest_node = (Tree_Graph.data[id_node][0], Tree_Graph.data[id_node][1])
  return closest_node

In [ ]:
Tree_Graph = spatial.KDTree(list(G.nodes()))

In [ ]:
# словарь с переменными, которые в дальнейшем будут заполняться центроидами услуг из геодатафремов
a = {'houses' : [], 'place_of_worship': [], 'post_office': [], 'pharmacy': []}   
for c in a:
  b = gpd.read_file("/content/" + c + ".geojson")   # открываю файлы, загруженные в colab
  globals()['gdf_'+ c] = gpd.GeoDataFrame(b)   # превращаю в gdf 
  globals()['gdf_'+ c].to_crs(3857)            # устанавливаю систему координат Псевдомеркатор
  for d in globals()['gdf_'+ c]['geometry']:   # в каждом gdf обращаюсь построчно к столбцу "geometry"
    a[c].append(d.centroid)   # определяю координаты центроида полигона, который изначально указан в столбце "geomerty"

In [ ]:
d = ['houses', 'pharmacy', 'post_office', 'place_of_worship']   
for c in d:
  keys = [str(i) for i in range(1, (len(a[c]) + 1))]
  values = [j for j in a[c]]
  globals()[c+'_dict'] = {k: {'point': v} for (k,v) in zip(keys, values)}
  for i in globals()[c +'_dict']:
    closest_point = search_nearest_point(Tree_Graph, (globals()[c+'_dict'][i]['point'].x, globals()[c+'_dict'][i]['point'].y))
    globals()[c+'_dict'][i]['nearest_point'] = closest_point

In [ ]:
len(houses_dict)*len(place_of_worship_dict)

In [ ]:
houses_dict

In [ ]:
place_of_worship_dict

In [ ]:
Paths = {}
for h in houses_dict:
  for d in place_of_worship_dict:
    h_point = houses_dict[h]['nearest_point']
    p_point = place_of_worship_dict[d]['nearest_point']
    path = nx.dijkstra_path(G, h_point, p_point, weight = 'weight')
    distance = 0
    for i in range(len(path) - 1):
      a,b = Point(path[i]), Point(path[i+1])
      distance += a.distance(b) 
    Paths[(h_point, p_point)] = {'path': path, 'distance': distance, 'point': houses_dict[h]['point']}
    # print(h_point)

In [ ]:
Paths

In [ ]:
Paths1 = {}
for h in houses_dict:
  for d in pharmacy_dict:
    h_point = houses_dict[h]['nearest_point']
    p_point = pharmacy_dict[d]['nearest_point']
    path = nx.dijkstra_path(G, h_point, p_point, weight = 'weight')
    distance = 0
    for i in range(len(path) - 1):
      a,b = Point(path[i]), Point(path[i+1])
      distance += a.distance(b) 
    Paths1[(h_point, p_point)] = {'path': path, 'distance': distance, 'point': houses_dict[h]['point']}
    # print(h_point)

In [ ]:
Paths1

In [ ]:
Paths2 = {}
for h in houses_dict:
  for d in post_office_dict:
    h_point = houses_dict[h]['nearest_point']
    p_point = post_office_dict[d]['nearest_point']
    path = nx.dijkstra_path(G, h_point, p_point, weight = 'weight')
    distance = 0
    for i in range(len(path) - 1):
      a,b = Point(path[i]), Point(path[i+1])
      distance += a.distance(b) 
    Paths2[(h_point, p_point)] = {'path': path, 'distance': distance, 'point': houses_dict[h]['point']}
    # print(h_point)

In [ ]:
Paths2

In [ ]:
# нужно произвести замену с a[0][0], на houses_dict['point']
data = Paths.copy()
parsed_data = [[(data[a]['point'].x, data[a]['point'].y), data[a]['distance']] for a in data]
df_1 = pd.DataFrame(data=parsed_data, columns=['coord', 'place_of_worship'])
df_1 = df_1.groupby(['coord']).min()
df_1

In [ ]:
pd.reset_option('display.max_rows', 9999)

In [ ]:
data1 = Paths1.copy()
parsed_data1 = [[(a[0][0], a[0][1]), data1[a]['distance']] for a in data1]
df2= pd.DataFrame(data=parsed_data1, columns=['coord', 'pharmacy'])
df2 = df2.groupby(['coord']).min()
df2

In [ ]:
data2 = Paths2.copy()
parsed_data2 = [[(a[0][0], a[0][1]), data2[a]['distance']] for a in data2]
df3= pd.DataFrame(data=parsed_data2, columns=['coord', 'post_office'])
df3 = df3.groupby(['coord']).min()
df3

In [ ]:
df4 = df1.merge(df2, how='inner', on='coord')
df4 = df4.reset_index()
df4 = df4.merge(df3, how='inner', on='coord')
df4

In [ ]:
# создаю геодатафрейм с заранее заданной системой координат из центроидов домов и наименьших расстояний до услуг
gdf = gpd.GeoDataFrame({'geometry': [Point(i) for i in df4['coord']], 'place_of_worship': df4['place_of_worship'], 
                        'post_office': df4['post_office'], 'pharmacy': df4['pharmacy']}, crs = "EPSG:3857")
gdf.insert(0, 'index', gdf.index.tolist())   # добавляю столбец с индексами, 
# # тк он понадобится мне при построении графиков. При этом индексы должны быть отдельным столбцом
gdf['dist'] = (gdf[['pharmacy', 'post_office', 'place_of_worship']].mean(axis = 1)).apply(lambda x: x*100000)   
# # считаю среднее расстояние от центроида дома до услуг + превращаю в метры
gdf = gdf.drop(['pharmacy', 'post_office', 'place_of_worship'], axis = 1)
gdf

In [ ]:
# параметры сетки, в каждом квадрате которой будем определять доступность
minX, minY, maxX, maxY = 30.209141, 59.945586, 30.335134, 59.982592    # поменял оси местами, тк сетка и точки в разных углах располагаются
step = 0.003   # шаг сетки 300 метров, деленный на 100000 для изменения исходных координат в тысячах километров

In [ ]:
# создание сетки квадратов
Grid = []
for x0 in np.arange(minX, maxX + step, step):
  for y0 in np.arange(minY, maxY + step, step):
    x1 = x0 - step
    y1 = y0 + step
    Grid.append(shapely.geometry.box(x0, y0, x1, y1))
grid = gpd.GeoDataFrame(Grid, columns = ['geometry'], crs = 'EPSG:3857')
grid

In [ ]:
# Координатная сетка и центроиды домов, раскрашенные в соответствии с расстоянием от них до ближайшей услуги 
ax = gdf.plot(figsize = (10,10), column = 'dist', cmap = 'jet', legend = True, 
              legend_kwds = {'label': 'Среднее расстояние до ближайшей услуги, м', 'orientation':'horizontal'})
grid.plot(ax=ax, facecolor="none", edgecolor='blue')
plt.savefig('center_grid.png')

In [ ]:
merged = gpd.sjoin(gdf, grid, how = 'left', predicate = 'within')   
# sjoin объединяет 2 гдф, left - центроиды домов (основа объединения), 
# предикатор within - точки будут внутри полигона 
merged = merged.drop(['index'], axis = 1)
merged

In [ ]:
diss = merged.dissolve(by = 'index_right', aggfunc = 'mean')  
# объединяет в единую геометрию, агрегирует все строки по какому-то признаку с использованием функции
diss

In [ ]:
# выбирает из всего grid строки, совпадающие по индексу с diss.index (diss['index_right'] == diss.index), и вписать туда значение dist из diss
grid.loc[diss.index, 'dist'] = diss.dist.values

In [ ]:
# визуализируем полученные результаты в виде ячеек сетки с цветом заливки в зависимости от расстояния до ближайших услуг
ax = grid.plot(figsize = (10,10), column = 'dist', cmap = 'jet', legend = True, legend_kwds = {'label': 'Среднее расстояние до ближайшей услуги, м', 
                                                                                              'orientation':'horizontal'})
plt.savefig('mean_dist.png')

In [ ]:
# скачиваю границы Петроградского района из osm
TERITORY_NAME = 'Петроградский район, Санкт-Петербург'
territory = ox.geocode_to_gdf(TERITORY_NAME)

In [ ]:
# рисую общую диаграмму обеспеченности услугами пенсионеров Петроградского района Санкт-Петербурга
map = grid.plot(figsize=(10,10), column = 'dist', cmap = 'jet', legend = True, legend_kwds = {'label': 'Среднее расстояние до ближайшей услуги, м', 
                                                                                              'orientation':'horizontal'})
# cmap - расцветка, градация цвета на основе dist

territory.plot(ax=map, color='none', edgecolor='black')  # рисую границы района, ax - указание на другой график, который также надо отобразить
ax.axis('on')  # визуализировать оси и их значение
plt.title('Картограмма обеспеченности услугами пенсионеров Петроградского района Санкт-Петербурга')
plt.savefig('diag.png')

In [ ]:
b = gpd.read_file("/content/gdf_dist")   # открываю файлы, загруженные в colab
gdf = gpd.GeoDataFrame(b)   # превращаю в gdf 
gdf.to_crs(3857) 
gdf = gdf.rename(columns = {'dist': 'Среднее расстояние от жилого дома до услуг по категориям, м'})

In [ ]:
gdf.explore(tiles = 'CartoDB positron', column = 'Среднее расстояние от жилого дома до услуг по категориям, м', cmap = 'Set2')

In [ ]:
b = gpd.read_file("/content/grid_dist")   # открываю файлы, загруженные в colab
grid = gpd.GeoDataFrame(b)   # превращаю в gdf 
grid.to_crs(3857) 
grid = grid.rename(columns = {'dist': 'Среднее расстояние от жилого дома до услуг по категориям, м'})

In [ ]:
grid

In [ ]:
grid.crs

In [ ]:
grid.explore(tiles = 'CartoDB positron', column = 'Среднее расстояние от жилого дома до услуг по категориям, м', cmap = 'Set2')

In [ ]:
pd.set_option('display.max_columns', 100)

In [ ]:
# тестирую geocode через геокодер osm
geocode("г. Санкт-Петербург, линия. 10-я В.О., д. 5", provider="arcgis", user_agent="my-application", timeout=None)
# nominatim - геокодер от osm, user-agent - имя устройства, timeout - время работы функции

In [ ]:
data = pd.read_csv('Tehniko-ekonomicheskie-pasporta-mnogokvartirnyh-domov.csv')
data = data[data['Район'] == 'Петроградский']
data = data[~data['Количество проживающих'].isnull()]

In [ ]:
d = data.copy()
d['coord'] = ''
d['coord'] = d.apply(lambda x: geocode(x.Адрес, provider="arcgis", user_agent="my-application", timeout=None)['geometry'], axis = 1)

In [ ]:
d['lon'] = ''
d['lat'] = ''
d['lon'] = d.apply(lambda x: x.geometry.x, axis = 1)
d['lat'] = d.apply(lambda x: x.geometry.y, axis = 1)

In [ ]:
# geoc = gpd.GeoDataFrame(d, geometry = 'coord', crs = 'EPSG:4326')

In [ ]:
geoc.to_file('population.geojson', driver="GeoJSON")  

In [ ]:
map = KeplerGl(height = 400)

In [ ]:
# m.explore(tiles = 'CartoDB positron', column = 'Количество проживающих', cmap = 'Set2')

In [ ]:
map.add_data(data = geoc.copy())

In [ ]:
map()

ЯНДЕКС ГЕОКОДИРОВАНИЕ

In [ ]:
!pip install yandex_geocoder

In [ ]:
from yandex_geocoder import Client
client = Client('525704ed-69ba-4e6f-b47b-abe45a2403c4')

In [ ]:
geoc['lat'] = ''
geoc['lon'] = ''
geoc['coord'] = ''
for i in range(len(geoc)):
  if i <= 980:
    address = list(geoc['Адрес'])[i]
    coordinates = client.coordinates(address)

    lon, lat = float(coordinates[0]), float(coordinates[1])
    geoc['lat'][i] = lat
    geoc['lon'][i] = lon
  else:
    continue

geoc['coord'] = ''
ge = geoc.copy()
ge['lon'] = ge['lon'].replace('', np.nan)
ge['lat'] = ge['lat'].replace('', np.nan)
ge['coord'] = ge.apply(lambda x: Point(x.lon, x.lat), axis = 1)